# <center> Patient Readmittance

## Data Fields
- `race Values`: Caucasian, Asian, African American, Hispanic, and other
- `gender Values`: male, female, and unknown/invalid
- `age` Grouped in 10-year intervals: 0, 10), 10, 20), …, 90, 100)
- `weight` Weight in pounds.
- `admission_type_id` Integer identifier corresponding to 9 distinct values, for example, emergency, urgent, elective, newborn, and not available
- `discharge_disposition_id` Integer identifier corresponding to 29 distinct values, for example, discharged to home, expired, and not available
- `admission_source_id` Integer identifier corresponding to 21 distinct values, for example, physician referral, emergency room, and transfer from a hospital
- `time_in_hospital` Integer number of days between admission and discharge
- `payer_code` Integer identifier corresponding to 23 distinct values, for example, Blue Cross/Blue Shield, Medicare, and self-pay
- `medical_specialty` Integer identifier of a specialty of the admitting physician, corresponding to 84 distinct values, for example, cardiology, internal medicine, family/general practice, and surgeon
- `num_lab_procedures` Number of lab tests performed during the encounter
- `num_procedures` Number of procedures (other than lab tests) performed during the encounter
- `num_medications` Number of distinct generic names administered during the encounter
- `number_outpatient` visits Number of outpatient visits of the patient in the year preceding the encounter
- `number_emergency` visits Number of emergency visits of the patient in the year preceding the encounter
- `number_inpatient` visits Number of inpatient visits of the patient in the year preceding the encounter
- `diag_1` The primary diagnosis (coded as first three digits of ICD9); 848 distinct values
- `diag_2` Secondary diagnosis (coded as first three digits of ICD9); 923 distinct values
- `diag_3` Additional secondary diagnosis (coded as first three digits of ICD9); 954 distinct values
- `number_diagnoses` Number of diagnoses entered to the system
- `max_glu_serum` Indicates the range of the result or if the test was not taken. Values: “>200,” “>300,” “normal,” and “none” if not measured
- `A1Cresult` Indicates the range of the result or if the test was not taken. Values: “>8” if the result was greater than 8%, “>7” if the result was greater than 7% but less than 8%, “normal” if the result was less than 7%, and “none” if not measured.
- For the generic names: `metformin, repaglinide, nateglinide, chlorpropamide, glimepiride, acetohexamide, glipizide, glyburide, tolbutamide, pioglitazone, rosiglitazone, acarbose, miglitol, troglitazone, tolazamide, examide, citoglipton, insulin, glyburide-metformin, glipizide-metformin, glimepiride-pioglitazone, metformin-rosiglitazone, and metformin-pioglitazone`, the feature indicates whether the drug was prescribed or there was a change in the dosage. Values: “up” if the dosage was increased during the encounter, “down” if the dosage was decreased, “steady” if the dosage did not change, and “no” if the drug was not prescribed
- `change` Indicates if there was a change in diabetic medications (either dosage or generic name). Values: “change” and “no change”
- `diabetesMed` Indicates if there was any diabetic medication prescribed. Values: “yes” and “no”
- `readmitted` Days to inpatient readmission. Values: “2” if the patient was readmitted in less than 30 days, “1” if the patient was readmitted in more than 30 days, and “0” for no record of readmission.

### Imporing Packages

In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import matplotlib.pyplot as plt # Visualization
import seaborn as sns # Visualization
colors = ['#B90276','#50237F', '#005691', '#008ECF','#E20015', '#00A8B0', '#78BE20', '#006249', '#525F6B']

sns.set_palette(sns.color_palette(colors))

import warnings
warnings.filterwarnings('ignore')

## Loading Data

In [2]:
import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/surya1610/601-Project-2-dataset/main/diabetic_data.csv")

### Below We can see that Features are mix of category and Numeric

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   encounter_id              101766 non-null  int64 
 1   patient_nbr               101766 non-null  int64 
 2   race                      101766 non-null  object
 3   gender                    101766 non-null  object
 4   age                       101766 non-null  object
 5   weight                    101766 non-null  object
 6   admission_type_id         101766 non-null  int64 
 7   discharge_disposition_id  101766 non-null  int64 
 8   admission_source_id       101766 non-null  int64 
 9   time_in_hospital          101766 non-null  int64 
 10  payer_code                101766 non-null  object
 11  medical_specialty         101766 non-null  object
 12  num_lab_procedures        101766 non-null  int64 
 13  num_procedures            101766 non-null  int64 
 14  num_

### Lets check summary of Numerical features

Below we can see the count, mean, standard deviation, max etc.


In [4]:
df.describe().T.style.set_properties(**{'background-color': 'white','color': 'black','border-color': 'black'})

ImportError: Missing optional dependency 'Jinja2'. DataFrame.style requires jinja2. Use pip or conda to install Jinja2.

### Lets have a look on the Unique count of each column

In [ ]:
df.nunique()

### Missing Values

#### As we can see below that no missing values are seen but missing values are in the form of question mark here

In [ ]:
ax=sns.heatmap(df.isnull(),yticklabels=False)

### So lets check the actual missing Values
As we can see below some columns has massive missing data

In [ ]:
for col in df.columns:
    if df[col].dtype == object:
         print(col,df[col][df[col] == '?'].count())

### Encoding the outcome variable
The outcome we are looking at is whether the patient gets readmitted to the hospital within 30 days or not. The variable actually has < 30, > 30 and No Readmission categories. We combined the readmission after 30 days and no readmission into a single category

In [ ]:
# 0 means not not admitted
# 1 means readmitted
df['readmitted'] = df['readmitted'].replace('>30', 0)
df['readmitted'] = df['readmitted'].replace('<30', 1)
df['readmitted'] = df['readmitted'].replace('NO', 0)

### Lets drop those Columns

In [ ]:
df = df.drop(['weight', 'payer_code', 'medical_specialty'], axis = 1)
df = df.drop(['encounter_id'], axis = 1)
df = df.drop(['citoglipton', 'examide'], axis = 1)

## Lets see some Class Distribution

### We can see that the data is not evenly distributed among Classes

In [ ]:
plt.figure(figsize=(7,7))
sns.countplot(data=df,x="readmitted")

### Below we can see that the data collected has almost equal gender distribution for male and female

In [ ]:
plt.figure(figsize=(10,5))
sns.countplot(df['gender']).set_title('Count of gender')

In [ ]:
plt.figure(figsize=(10,5))
new_df=df[df['readmitted']==1]
sns.countplot(new_df['age'], hue = new_df['readmitted'],order = new_df['age'].value_counts().index).set_title('Count of age')

### There doesn't seem any high correlation among the features

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
f,ax = plt.subplots(figsize=(8, 6))
sns.heatmap(df.corr(), annot=True, linewidths=0.5,linecolor="black", fmt= '.2f',ax=ax,cmap="coolwarm")
plt.show()

## More Data Cleaning
converting all categorical values into numerical data for better visualization

In [ ]:
dff = pd.read_csv("https://raw.githubusercontent.com/surya1610/601-Project-2-dataset/main/diabetic_data.csv")
for i in range(24,47): #No-0,Down-1,Up-2,Steady-3
    dff[dff.columns[i]] = dff[dff.columns[i]].map({'No': -2, 'Down': -1, "Up":1, 'Steady':0})
dff['readmitted'] = dff['readmitted'].map({'NO': 0, '<30': 1, ">30":0})
dff['readmittedbinary'] = dff['readmitted'].map({0: 0, 1: 1})
dff['change'] = dff['change'].map({'No': -1, 'Ch': 1})#No-0,Ch-1
dff['max_glu_serum'] = dff['max_glu_serum'].map({'None': 0, '>200': 2,'>300':3,'Norm':1})
dff['A1Cresult'] = dff['A1Cresult'].map({'None': 0, '>7': 7,'>8':8,'Norm':5})
dff['diabetesMed'] = dff['diabetesMed'].map({'No': -1, 'Yes': 1})
dff['age'] = dff['age'].map({'[0-10)':5,'[10-20)':15, '[20-30)':25,'[30-40)':35,'[40-50)':45,'[50-60)':55,'[60-70)':65,'[70-80)':75,'[80-90)':85,'[90-100)':95})
dff.drop(['encounter_id','patient_nbr','weight','admission_type_id','discharge_disposition_id','admission_source_id','medical_specialty','payer_code'],axis=1,inplace=True)
dff=dff.loc[dff['gender'].isin(['Male','Female'])]
dff.replace('?', np.nan, inplace = True)
dff= dff.dropna()


In [ ]:
new_df=dff[dff['readmitted']==1]

### Number of Medications are almost Same for all

In [ ]:
fig = plt.figure(figsize=(8,8))
sns.barplot(x = dff['readmitted'], y = dff['num_medications']).set_title("Number of medication used VS. Readmission")

## below we can see that females are more who suffer with Diabetes and get Readmitted

In [ ]:
fig = plt.figure(figsize=(8,8))
sns.countplot(x = new_df['gender'], hue = new_df['readmitted']).set_title("Gender of Patient VS. Readmission")

## Change in Medication doesn't gives a clear indication about Readmission

In [ ]:
fig = plt.figure(figsize=(8,8))
sns.countplot(dff['change'], hue = dff['readmitted']).set_title('Change of Medication VS. Readmission')

### Below we can see that, If the patient is prescribed some Medication still he has to visit or in most of the cases Patients are Readmitted

In [ ]:
fig = plt.figure(figsize=(8,8))
sns.countplot(dff['diabetesMed'], hue = dff['readmitted']).set_title('Diabetes Medication prescribed VS Readmission')

## Question 1

AfricanAmericans are tend to suffer with diabetes and get readmitted?

In [ ]:
c = sns.countplot(x = new_df['race'], hue= new_df['readmitted'])

 NO,
 Our assumption is wrong in this case as Caucasians are the one who Mostly suffers with Diabetes

# Question 2

 Chances for getting readmitted for the age group >40?

In [ ]:
count_of_y = dff["age"].groupby(dff["readmitted"]).value_counts().rename("counts").reset_index()
fig = sns.lineplot(x="age", y="counts", hue="readmitted", data=count_of_y)

In [ ]:
fig = plt.figure(figsize=(15,10))
sns.countplot(x= new_df['age'], hue = new_df['readmitted']).set_title('Age of Patient VS. Readmission')

YES, Here we can see Higher the Persons age , Higher is the chance to get Diabetes

# Question 3

Patients who spend more time in the hospital tend to have higher chances to get readmitted?

In [ ]:
import matplotlib.patches as mpatches
print("Red: Readmitted")
print("Blue: Not Readmitted")
fig = plt.figure(figsize=(13,7),)
ax=sns.kdeplot(dff.loc[(dff['readmitted'] == 0),'time_in_hospital'] , color='b',shade=True,label='Not Readmitted')
ax=sns.kdeplot(dff.loc[(dff['readmitted'] == 1),'time_in_hospital'] , color='r',shade=True, label='Readmitted')
red_patch = mpatches.Patch(color='red', label='Readmitted',)
blue_patch = mpatches.Patch(color='blue', label='Not Readmitted',)
ax.set(xlabel='Time in Hospital', ylabel='Frequency')
plt.title('Time in Hospital VS. Readmission')

No, The more the time patients spend in hospital, the less they get readmitted according to the above analysis

## Question 4

Number of Lab Procedures for the patients are directly related to readmission?

In [ ]:
fig = plt.figure(figsize=(15,6),)
ax=sns.kdeplot(dff.loc[(dff['readmitted'] == 0),'num_lab_procedures'] , color='b',shade=True,label='Not readmitted')
ax=sns.kdeplot(dff.loc[(dff['readmitted'] == 1),'num_lab_procedures'] , color='r',shade=True, label='readmitted')
ax.set(xlabel='Number of lab procedure', ylabel='Frequency')
plt.title('Number of lab procedure VS. Readmission')

YES, Below we can see that the graphs plotted for lab procedures and readmissions are similar which proves our hypothesis

# Conclusion

As the Data is ***Sparse*** and Even due to the less correlation its been Difficult to map the features.
And most of the features are objects. outlier treatment seems no use using this data.

For the given data  Below features are classified as **Important Features**

'time_in_hospital', <br>
 'num_lab_procedures', <br>
 'num_medications', <br>
 'number_emergency', <br>
 'number_inpatient', <br>
 'diag_1', <br>
 'diag_2', <br>
 'diag_3', <br>
 'number_diagnoses', <br>
 'metformin_index_vector',